In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [3]:
df = pd.read_csv(r"C:\Users\kotta\OneDrive\Desktop\Restaurant-recommender\zomato (2).csv",encoding='latin-1')

print(df.head())

   Restaurant ID                        Restaurant Name Country   City  \
0        3400025                             Jahanpanah    India  Agra   
1        3400341                    Rangrezz Restaurant    India  Agra   
2        3400005                Time2Eat - Mama Chicken    India  Agra   
3        3400021  Chokho Jeeman Marwari Jain Bhojanalya    India  Agra   
4        3400017                         Pinch Of Spice    India  Agra   

                                             Address     Locality  \
0  E 23, Shopping Arcade, Sadar Bazaar, Agra Cant...   Agra Cantt   
1  E-20, Shopping Arcade, Sadar Bazaar, Agra Cant...   Agra Cantt   
2        Main Market, Sadar Bazaar, Agra Cantt, Agra   Agra Cantt   
3  1/48, Delhi Gate, Station Road, Raja Mandi, Ci...  Civil Lines   
4  23/453, Opposite Sanjay Cinema, Wazipura Road,...  Civil Lines   

    Locality Verbose  Longitude   Latitude                        Cuisines  \
0   Agra Cantt, Agra  78.011544  27.161661           North Ind

In [4]:
from numpy import astype
df['combined_features'] = (
    df['Cuisines'].astype(str) + " " +
    df['City'].astype(str) + " " +
    df['Locality'].astype(str) + " " +
    df['Aggregate rating'].astype(str) + " " +
    df['Average Cost for two'].astype(str)
)

In [5]:
vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["combined_features"])

In [6]:
def recommend_by_query(query, city=None, max_cost=None, top_n=5):

    query_vec = vectorizer.transform([query])
    similarity = cosine_similarity(query_vec, tfidf_matrix)

    scores = list(enumerate(similarity[0]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    indices = [i[0] for i in scores]

    result = df.iloc[indices]

    if city:
        result = result[result['City'].str.lower() == city.lower()]

    if max_cost:
        result = result[result['Average Cost for two'] <= max_cost]

    return result.head(top_n)[
        ['Restaurant Name', 'Cuisines', 'City', 'Locality',
         'Average Cost for two', 'Aggregate rating']
    ]


In [7]:
recommend_by_query("biryani", city="Hyderabad",max_cost=500)

,Restaurant Name,Cuisines,City,Locality,Average Cost for two,Aggregate rating
1691,Churrolto,"Desserts, Cafe, Mexican",Hyderabad,Madhapur,500,4.7


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder

# Create a numerical label from 'Rating text' for classification
le = LabelEncoder()
df['rating_label_encoded'] = le.fit_transform(df['Rating text'])

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df['combined_features']) # Use combined_features as text input
y = df['rating_label_encoded'] # Use the new numerical label as target

model = LogisticRegression(max_iter=1000) # Increased max_iter for convergence
model.fit(X, y)

LogisticRegression(max_iter=1000)

In [9]:
import pickle
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(vectorizer, open('vectorizer.pkl', 'wb'))

In [12]:
model = pickle.load(open('model.pkl','rb'))
vectorizer = pickle.load(open('vectorizer.pkl','rb'))
tfidf_matrix = vectorizer.fit_transform(df['Cuisines'])
pickle.dump(tfidf_matrix, open("tfidf_matrix.pkl", "wb"))
